# MADAD — 05 CatBoost Quantile Forecasting
## Native Categorical Features — P10 / P50 / P90

Standalone CatBoost notebook.

CatBoost receives `hf_pk`, `productID`, `facility_type`, and `district` as native categorical features.

**Runs only:** shared leakage-safe preparation → CatBoost tuning/evaluation → save results.

**Test remains untouched.**


In [1]:
# ============================================================
# 1) Mount Drive and set paths
# ============================================================
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path

BASE_PATH = Path(
    '/content/drive/MyDrive/Madad for Medical Logistics/Dataset/Dryad_Dataset'
)

DATA_PATH = BASE_PATH / 'Madad_Integrated_Data.csv'

OUTPUT_PATH = (
    BASE_PATH
    / 'forecasting_outputs_clean'
    / '05_catboost_quantile'
)

OUTPUT_PATH.mkdir(
    parents=True,
    exist_ok=True
)

print('Data path:', DATA_PATH)
print('Exists:', DATA_PATH.exists())
print('Save folder:', OUTPUT_PATH)

# ============================================================
# 2) Install and import packages
# ============================================================
!pip -q install "catboost>=1.2" joblib

import warnings
warnings.filterwarnings('ignore')

import json
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    mean_pinball_loss,
    r2_score
)

from catboost import CatBoostRegressor

RANDOM_STATE = 42
EPS = 1e-6


Mounted at /content/drive
Data path: /content/drive/MyDrive/Madad for Medical Logistics/Dataset/Dryad_Dataset/Madad_Integrated_Data.csv
Exists: True
Save folder: /content/drive/MyDrive/Madad for Medical Logistics/Dataset/Dryad_Dataset/forecasting_outputs_clean/05_catboost_quantile
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 97.1/97.1 MB 9.8 MB/s eta 0:00:00


In [2]:
# ============================================================
# 3) Load integrated Madad data
# ============================================================
df = pd.read_csv(
    DATA_PATH,
    low_memory=False
)

df['date_parsed'] = pd.to_datetime(
    df['date_parsed'],
    errors='coerce'
)

group_cols = ['hf_pk', 'productID']

df_model = (
    df
    .dropna(
        subset=[
            'date_parsed',
            'hf_pk',
            'productID',
            'consumption'
        ]
    )
    .copy()
)

df_model = (
    df_model
    .sort_values(group_cols + ['date_parsed'])
    .reset_index(drop=True)
)

print('Integrated rows:', f'{len(df_model):,}')
print('Facilities:', df_model['hf_pk'].nunique())
print('Products:', df_model['productID'].nunique())
print(
    'Date range:',
    df_model['date_parsed'].min(),
    'to',
    df_model['date_parsed'].max()
)

print('\nConsumption summary:')
print(df_model['consumption'].describe())


Integrated rows: 457,225
Facilities: 1091
Products: 36
Date range: 2019-10-01 00:00:00 to 2023-11-01 00:00:00

Consumption summary:
count    457225.000000
mean        161.034016
std         378.274544
min           0.000000
25%          10.000000
50%          50.000000
75%         173.000000
max       48000.000000
Name: consumption, dtype: float64


In [3]:
# ============================================================
# 4) Calendar-safe helper functions
# ============================================================
def calendar_month_gap(current_date, previous_date):
    return (
        (current_date.dt.year - previous_date.dt.year) * 12
        + (current_date.dt.month - previous_date.dt.month)
    )


def add_calendar_safe_lags(
    data,
    group_cols,
    date_col,
    value_cols,
    lags
):
    data = data.copy()

    for col in value_cols:
        if col not in data.columns:
            continue

        grouped = data.groupby(
            group_cols,
            sort=False
        )

        for lag in lags:
            previous_value = grouped[col].shift(lag)
            previous_date = grouped[date_col].shift(lag)

            gap = calendar_month_gap(
                data[date_col],
                previous_date
            )

            data[f'{col}_lag_{lag}'] = (
                previous_value.where(gap == lag)
            )

    return data


def add_calendar_lagged_aggregate_features(
    base_df,
    group_keys,
    source_col,
    prefix,
    lags=range(1, 7),
    agg='mean'
):
    # Monthly group aggregate is built using only that month's rows.
    monthly = (
        base_df
        .groupby(
            group_keys + ['date_parsed'],
            as_index=False
        )[source_col]
        .agg(agg)
        .rename(columns={source_col: '_aggregate_value'})
    )

    result = base_df

    for lag in lags:
        lagged = monthly.copy()

        lagged['date_parsed'] = (
            lagged['date_parsed']
            + pd.offsets.MonthBegin(lag)
        )

        lagged = lagged.rename(
            columns={
                '_aggregate_value':
                f'{prefix}_lag_{lag}'
            }
        )

        result = result.merge(
            lagged[
                group_keys
                + ['date_parsed', f'{prefix}_lag_{lag}']
            ],
            on=group_keys + ['date_parsed'],
            how='left'
        )

    return result


In [4]:
# ============================================================
# 5) Local temporal features BEFORE target construction
# ============================================================
lag_source_cols = [
    c for c in [
        'consumption',
        'received',
        'openBalance',
        'closeBalance',
        'stockout'
    ]
    if c in df_model.columns
]

df_model = add_calendar_safe_lags(
    data=df_model,
    group_cols=group_cols,
    date_col='date_parsed',
    value_cols=lag_source_cols,
    lags=range(1, 13)
)

for col in [
    'consumption',
    'received',
    'closeBalance'
]:
    if col not in df_model.columns:
        continue

    for window, min_history in [
        (3, 2),
        (6, 3),
        (12, 6)
    ]:
        lag_cols = [
            f'{col}_lag_{lag}'
            for lag in range(1, window + 1)
            if f'{col}_lag_{lag}' in df_model.columns
        ]

        history_count = (
            df_model[lag_cols]
            .notna()
            .sum(axis=1)
        )

        df_model[
            f'{col}_history_count_{window}'
        ] = history_count

        df_model[
            f'{col}_roll_mean_{window}'
        ] = (
            df_model[lag_cols]
            .mean(axis=1, skipna=True)
            .where(history_count >= min_history)
        )

        df_model[
            f'{col}_roll_std_{window}'
        ] = (
            df_model[lag_cols]
            .std(axis=1, skipna=True)
            .where(history_count >= min_history)
        )

for window, min_history in [
    (3, 2),
    (6, 3),
    (12, 6)
]:
    lag_cols = [
        f'stockout_lag_{lag}'
        for lag in range(1, window + 1)
        if f'stockout_lag_{lag}' in df_model.columns
    ]

    history_count = (
        df_model[lag_cols]
        .notna()
        .sum(axis=1)
    )

    df_model[f'stockout_rate_past_{window}'] = (
        df_model[lag_cols]
        .mean(axis=1, skipna=True)
        .where(history_count >= min_history)
    )

df_model['year'] = df_model['date_parsed'].dt.year
df_model['month'] = df_model['date_parsed'].dt.month
df_model['quarter_num'] = df_model['date_parsed'].dt.quarter

df_model['month_sin'] = np.sin(
    2 * np.pi * df_model['month'] / 12
)

df_model['month_cos'] = np.cos(
    2 * np.pi * df_model['month'] / 12
)

if {
    'consumption_roll_mean_3',
    'consumption_roll_mean_6'
}.issubset(df_model.columns):
    df_model['demand_trend_3_vs_6'] = (
        df_model['consumption_roll_mean_3']
        - df_model['consumption_roll_mean_6']
    )

if {
    'consumption_roll_mean_3',
    'consumption_roll_mean_12'
}.issubset(df_model.columns):
    df_model['demand_trend_3_vs_12'] = (
        df_model['consumption_roll_mean_3']
        - df_model['consumption_roll_mean_12']
    )

if 'consumption_roll_mean_12' in df_model.columns:
    df_model['current_consumption_vs_12m'] = (
        df_model['consumption']
        / (
            df_model['consumption_roll_mean_12'].abs()
            + EPS
        )
    )

if {
    'closeBalance',
    'consumption_roll_mean_3'
}.issubset(df_model.columns):
    df_model['inventory_runway_3m'] = (
        df_model['closeBalance']
        / (
            df_model['consumption_roll_mean_3'].abs()
            + EPS
        )
    )

if {
    'received',
    'consumption'
}.issubset(df_model.columns):
    df_model['received_to_consumption'] = (
        df_model['received']
        / (
            df_model['consumption'].abs()
            + EPS
        )
    )

df_model['facility_product_observation_count_prior'] = (
    df_model
    .groupby(group_cols)
    .cumcount()
)

print('Local temporal features complete.')


Local temporal features complete.


In [5]:
# ============================================================
# 6) NEW V2 CONTEXT:
# Lagged product-level and district-product demand signals
# ============================================================
# These are all shifted forward by explicit calendar months.
# Therefore, a current row only receives aggregate demand
# information from prior calendar months.

df_model = add_calendar_lagged_aggregate_features(
    base_df=df_model,
    group_keys=['productID'],
    source_col='consumption',
    prefix='product_mean_consumption',
    lags=range(1, 7),
    agg='mean'
)

if 'district' in df_model.columns:
    df_model = add_calendar_lagged_aggregate_features(
        base_df=df_model,
        group_keys=['district', 'productID'],
        source_col='consumption',
        prefix='district_product_mean_consumption',
        lags=range(1, 7),
        agg='mean'
    )

for prefix in [
    'product_mean_consumption',
    'district_product_mean_consumption'
]:
    lag3 = [
        f'{prefix}_lag_{lag}'
        for lag in range(1, 4)
        if f'{prefix}_lag_{lag}' in df_model.columns
    ]

    lag6 = [
        f'{prefix}_lag_{lag}'
        for lag in range(1, 7)
        if f'{prefix}_lag_{lag}' in df_model.columns
    ]

    if lag3:
        df_model[f'{prefix}_roll3'] = (
            df_model[lag3]
            .mean(axis=1, skipna=True)
        )

    if lag6:
        df_model[f'{prefix}_roll6'] = (
            df_model[lag6]
            .mean(axis=1, skipna=True)
        )

    if (
        f'{prefix}_roll3' in df_model.columns
        and f'{prefix}_roll6' in df_model.columns
    ):
        df_model[f'{prefix}_trend_3_vs_6'] = (
            df_model[f'{prefix}_roll3']
            - df_model[f'{prefix}_roll6']
        )

print('Contextual aggregate features complete.')


Contextual aggregate features complete.


In [6]:
# ============================================================
# EXPERIMENT 2 — Demand-dynamics features
# ============================================================
def safe_ratio(num, den):
    return num / (np.abs(den) + EPS)

if 'consumption_lag_1' in df_model.columns:
    df_model['consumption_growth_1m'] = df_model['consumption'] - df_model['consumption_lag_1']
    df_model['consumption_growth_rate_1m'] = safe_ratio(df_model['consumption'] - df_model['consumption_lag_1'], df_model['consumption_lag_1'])

if {'consumption_roll_mean_3','consumption_roll_mean_6'}.issubset(df_model.columns):
    df_model['recent_vs_6m_ratio'] = safe_ratio(df_model['consumption_roll_mean_3'], df_model['consumption_roll_mean_6'])

if {'consumption_roll_mean_3','consumption_roll_mean_12'}.issubset(df_model.columns):
    df_model['recent_vs_12m_ratio'] = safe_ratio(df_model['consumption_roll_mean_3'], df_model['consumption_roll_mean_12'])

for window in [3,6,12]:
    mean_col=f'consumption_roll_mean_{window}'
    std_col=f'consumption_roll_std_{window}'
    if {mean_col,std_col}.issubset(df_model.columns):
        df_model[f'consumption_cv_{window}m'] = safe_ratio(df_model[std_col], df_model[mean_col])
    lag_cols=[f'consumption_lag_{lag}' for lag in range(1,window+1) if f'consumption_lag_{lag}' in df_model.columns]
    if lag_cols:
        valid_count=df_model[lag_cols].notna().sum(axis=1)
        zero_count=df_model[lag_cols].eq(0).sum(axis=1)
        df_model[f'zero_consumption_rate_{window}m'] = zero_count / valid_count.replace(0,np.nan)

if 'consumption_roll_mean_6' in df_model.columns:
    df_model['demand_spike_ratio'] = safe_ratio(df_model['consumption'], df_model['consumption_roll_mean_6'])

if {'consumption_lag_1','consumption_lag_3'}.issubset(df_model.columns):
    df_model['consumption_momentum_3m'] = df_model['consumption_lag_1'] - df_model['consumption_lag_3']

print('Experiment 2 demand-dynamics features added.')

Experiment 2 demand-dynamics features added.


In [7]:
# ============================================================
# 7) Construct NEXT-CALENDAR-MONTH demand target
# ============================================================
grouped = df_model.groupby(
    group_cols,
    sort=False
)

df_model['next_date_parsed'] = (
    grouped['date_parsed']
    .shift(-1)
)

df_model['next_month_consumption_observed'] = (
    grouped['consumption']
    .shift(-1)
)

df_model['month_start'] = (
    df_model['date_parsed']
    .dt.to_period('M')
    .dt.to_timestamp()
)

df_model['next_month_start'] = (
    df_model['next_date_parsed']
    .dt.to_period('M')
    .dt.to_timestamp()
)

df_model['expected_next_month'] = (
    df_model['month_start']
    + pd.offsets.MonthBegin(1)
)

is_true_next_month = (
    df_model['next_month_start']
    == df_model['expected_next_month']
)

df_model['target_consumption_next_month'] = (
    np.where(
        is_true_next_month,
        df_model['next_month_consumption_observed'],
        np.nan
    )
)

df_model = (
    df_model
    .dropna(
        subset=['target_consumption_next_month']
    )
    .copy()
)

target_gap = calendar_month_gap(
    df_model['next_month_start'],
    df_model['month_start']
)

assert (target_gap == 1).all()

print(
    'Eligible forecasting rows:',
    f'{len(df_model):,}'
)

print(
    'Target relation:',
    'ALL rows are exactly next calendar month.'
)

print('\nTarget summary:')
print(
    df_model[
        'target_consumption_next_month'
    ].describe()
)


Eligible forecasting rows: 313,698
Target relation: ALL rows are exactly next calendar month.

Target summary:
count    313698.000000
mean        167.782345
std         400.037253
min           0.000000
25%          10.000000
50%          50.000000
75%         195.000000
max       48000.000000
Name: target_consumption_next_month, dtype: float64


In [8]:
# ============================================================
# 8) Chronological whole-month split
# ============================================================
df_model = (
    df_model
    .sort_values('date_parsed')
    .reset_index(drop=True)
)

unique_dates = np.array(
    sorted(
        df_model[
            'date_parsed'
        ].dropna().unique()
    )
)

n_dates = len(unique_dates)

train_cut = (
    max(
        1,
        int(n_dates * 0.715)
    )
    - 1
)

val_cut = (
    max(
        train_cut + 1,
        int(
            n_dates
            * (0.715 + 0.127)
        )
    )
    - 1
)

train_end_date = pd.Timestamp(
    unique_dates[train_cut]
)

val_end_date = pd.Timestamp(
    unique_dates[
        min(
            val_cut,
            n_dates - 1
        )
    ]
)

train_df = df_model[
    df_model['date_parsed']
    <= train_end_date
].copy()

val_df = df_model[
    (
        df_model['date_parsed']
        > train_end_date
    )
    & (
        df_model['date_parsed']
        <= val_end_date
    )
].copy()

test_df = df_model[
    df_model['date_parsed']
    > val_end_date
].copy()

print(
    'Train:',
    f'{len(train_df):,}',
    train_df['date_parsed'].min(),
    'to',
    train_df['date_parsed'].max()
)

print(
    'Validation:',
    f'{len(val_df):,}',
    val_df['date_parsed'].min(),
    'to',
    val_df['date_parsed'].max()
)

print(
    'Test:',
    f'{len(test_df):,}',
    test_df['date_parsed'].min(),
    'to',
    test_df['date_parsed'].max()
)

print(
    'Total:',
    f'{len(train_df) + len(val_df) + len(test_df):,}'
)

assert (
    train_df['date_parsed'].max()
    < val_df['date_parsed'].min()
)

assert (
    val_df['date_parsed'].max()
    < test_df['date_parsed'].min()
)


Train: 211,452 2019-10-01 00:00:00 to 2022-08-01 00:00:00
Validation: 39,864 2022-09-01 00:00:00 to 2023-02-01 00:00:00
Test: 62,382 2023-03-01 00:00:00 to 2023-10-01 00:00:00
Total: 313,698


In [9]:
# ============================================================
# 9) Evaluation helpers
# ============================================================
TARGET = 'target_consumption_next_month'

def regression_metrics(
    y_true,
    y_pred
):
    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred)

    mae = mean_absolute_error(
        y_true,
        y_pred
    )

    rmse = np.sqrt(
        mean_squared_error(
            y_true,
            y_pred
        )
    )

    r2 = r2_score(
        y_true,
        y_pred
    )

    denominator = np.sum(
        np.abs(y_true)
    )

    wape = (
        np.sum(
            np.abs(
                y_true - y_pred
            )
        )
        / denominator
        if denominator > 0
        else np.nan
    )

    return {
        'MAE': mae,
        'RMSE': rmse,
        'WAPE': wape,
        'R2': r2
    }


def print_metrics(
    name,
    y_true,
    y_pred
):
    metrics = regression_metrics(
        y_true,
        y_pred
    )

    print(f'--- {name} ---')
    print(
        f"MAE : "
        f"{metrics['MAE']:.4f}"
    )
    print(
        f"RMSE: "
        f"{metrics['RMSE']:.4f}"
    )
    print(
        f"WAPE: "
        f"{metrics['WAPE']*100:.2f}%"
    )
    print(
        f"R²  : "
        f"{metrics['R2']:.4f}"
    )

    return metrics


In [10]:
# ============================================================
# 11) Select leakage-safe forecasting features
# ============================================================
base_numeric = [
    'consumption_growth_1m',
    'consumption_growth_rate_1m',
    'recent_vs_6m_ratio',
    'recent_vs_12m_ratio',
    'consumption_cv_3m',
    'consumption_cv_6m',
    'consumption_cv_12m',
    'zero_consumption_rate_3m',
    'zero_consumption_rate_6m',
    'zero_consumption_rate_12m',
    'demand_spike_ratio',
    'consumption_momentum_3m',

    'consumption',
    'received',
    'openBalance',
    'closeBalance',
    'normAvg',
    'normStd',
    'year',
    'month',
    'quarter_num',
    'month_sin',
    'month_cos',
    'facility_product_observation_count_prior',
    'demand_trend_3_vs_6',
    'demand_trend_3_vs_12',
    'current_consumption_vs_12m',
    'inventory_runway_3m',
    'received_to_consumption'
]

dynamic_features = [
    c for c in df_model.columns
    if any(
        c.startswith(prefix)
        for prefix in [
            'consumption_lag_',
            'received_lag_',
            'openBalance_lag_',
            'closeBalance_lag_',
            'stockout_lag_',
            'consumption_roll_',
            'received_roll_',
            'closeBalance_roll_',
            'consumption_history_count_',
            'received_history_count_',
            'closeBalance_history_count_',
            'stockout_rate_past_',
            'product_mean_consumption_',
            'district_product_mean_consumption_'
        ]
    )
]

numeric_features = [
    c
    for c in (
        base_numeric
        + dynamic_features
    )
    if c in train_df.columns
]

numeric_features = list(
    dict.fromkeys(
        numeric_features
    )
)

categorical_features = [
    c for c in [
        'hf_pk',
        'productID',
        'facility_type',
        'district'
    ]
    if c in train_df.columns
]

numeric_features = [
    c
    for c in numeric_features
    if not train_df[c].isna().all()
]

feature_cols = (
    numeric_features
    + categorical_features
)

print(
    'Numeric features:',
    len(numeric_features)
)

print(
    'Categorical features:',
    len(categorical_features)
)

print(
    'Total raw features:',
    len(feature_cols)
)


Numeric features: 137
Categorical features: 4
Total raw features: 141


In [11]:
# ============================================================
# EXPERIMENT 4A — Prepare native CatBoost feature matrices
# ============================================================
catboost_numeric_features = numeric_features.copy()
catboost_categorical_features = categorical_features.copy()
catboost_feature_cols = (
    catboost_numeric_features
    + catboost_categorical_features
)

X_train_cb = train_df[
    catboost_feature_cols
].copy()

X_val_cb = val_df[
    catboost_feature_cols
].copy()

# Numeric missing values
for col in catboost_numeric_features:
    median_value = X_train_cb[col].median()

    if pd.isna(median_value):
        median_value = 0.0

    X_train_cb[col] = (
        pd.to_numeric(
            X_train_cb[col],
            errors='coerce'
        )
        .fillna(median_value)
        .astype(float)
    )

    X_val_cb[col] = (
        pd.to_numeric(
            X_val_cb[col],
            errors='coerce'
        )
        .fillna(median_value)
        .astype(float)
    )

# CatBoost categorical columns should be strings with no missing values
for col in catboost_categorical_features:
    X_train_cb[col] = (
        X_train_cb[col]
        .fillna('__MISSING__')
        .astype(str)
    )

    X_val_cb[col] = (
        X_val_cb[col]
        .fillna('__MISSING__')
        .astype(str)
    )

cat_feature_indices = [
    X_train_cb.columns.get_loc(col)
    for col in catboost_categorical_features
]

print('CatBoost Train shape:', X_train_cb.shape)
print('CatBoost Validation shape:', X_val_cb.shape)
print('Native categorical features:', catboost_categorical_features)
print('Categorical indices:', cat_feature_indices)


CatBoost Train shape: (211452, 141)
CatBoost Validation shape: (39864, 141)
Native categorical features: ['hf_pk', 'productID', 'facility_type', 'district']
Categorical indices: [137, 138, 139, 140]


In [14]:
# ============================================================
# DEFINE TARGET ARRAYS FOR CATBOOST
# ============================================================

y_train = train_df[TARGET].astype(float).values
y_val = val_df[TARGET].astype(float).values

print('y_train shape:', y_train.shape)
print('y_val shape:', y_val.shape)

y_train shape: (211452,)
y_val shape: (39864,)


In [15]:
# ============================================================
# EXPERIMENT 4B — CatBoost P50 compact tuning
# ============================================================
cb_p50_candidates = [
    {
        'depth': 6,
        'learning_rate': 0.03,
        'l2_leaf_reg': 3.0,
        'random_strength': 1.0
    },
    {
        'depth': 7,
        'learning_rate': 0.03,
        'l2_leaf_reg': 3.0,
        'random_strength': 1.0
    },
    {
        'depth': 8,
        'learning_rate': 0.03,
        'l2_leaf_reg': 5.0,
        'random_strength': 1.0
    },
    {
        'depth': 7,
        'learning_rate': 0.05,
        'l2_leaf_reg': 5.0,
        'random_strength': 0.5
    },
    {
        'depth': 8,
        'learning_rate': 0.02,
        'l2_leaf_reg': 7.0,
        'random_strength': 0.5
    }
]

cb_p50_rows = []
cb_p50_models = {}

for i, params in enumerate(cb_p50_candidates, start=1):
    print(
        f'CatBoost P50 candidate {i}/{len(cb_p50_candidates)}:',
        params
    )

    model = CatBoostRegressor(
        loss_function='Quantile:alpha=0.5',
        eval_metric='Quantile:alpha=0.5',
        iterations=2000,
        random_seed=RANDOM_STATE,
        task_type='GPU',
        devices='0',
        verbose=False,
        allow_writing_files=False,
        **params
    )

    model.fit(
        X_train_cb,
        y_train,
        cat_features=cat_feature_indices,
        eval_set=(X_val_cb, y_val),
        early_stopping_rounds=80,
        verbose=False
    )

    pred = np.clip(
        model.predict(X_val_cb),
        0,
        None
    )

    metrics = regression_metrics(
        y_val,
        pred
    )

    pinball = mean_pinball_loss(
        y_val,
        pred,
        alpha=0.50
    )

    cb_p50_rows.append({
        'candidate': i,
        'pinball_loss': pinball,
        'best_iteration': model.get_best_iteration(),
        **metrics,
        **params
    })

    cb_p50_models[i] = model

cb_p50_results = (
    pd.DataFrame(cb_p50_rows)
    .sort_values(['pinball_loss', 'MAE'])
    .reset_index(drop=True)
)

display(cb_p50_results)

best_cb_p50_id = int(
    cb_p50_results.iloc[0]['candidate']
)

best_cb_p50_model = cb_p50_models[
    best_cb_p50_id
]

best_cb_p50_pred = np.clip(
    best_cb_p50_model.predict(X_val_cb),
    0,
    None
)

best_cb_p50_metrics = regression_metrics(
    y_val,
    best_cb_p50_pred
)

best_cb_p50_pinball = mean_pinball_loss(
    y_val,
    best_cb_p50_pred,
    alpha=0.50
)

print('\nBEST CATBOOST P50 CANDIDATE:', best_cb_p50_id)
print('Pinball Loss:', round(best_cb_p50_pinball, 4))
print('MAE:', round(best_cb_p50_metrics['MAE'], 4))
print('RMSE:', round(best_cb_p50_metrics['RMSE'], 4))
print('WAPE:', f"{best_cb_p50_metrics['WAPE']*100:.2f}%")
print('R²:', round(best_cb_p50_metrics['R2'], 4))


CatBoost P50 candidate 1/5: {'depth': 6, 'learning_rate': 0.03, 'l2_leaf_reg': 3.0, 'random_strength': 1.0}
CatBoost P50 candidate 2/5: {'depth': 7, 'learning_rate': 0.03, 'l2_leaf_reg': 3.0, 'random_strength': 1.0}
CatBoost P50 candidate 3/5: {'depth': 8, 'learning_rate': 0.03, 'l2_leaf_reg': 5.0, 'random_strength': 1.0}
CatBoost P50 candidate 4/5: {'depth': 7, 'learning_rate': 0.05, 'l2_leaf_reg': 5.0, 'random_strength': 0.5}
CatBoost P50 candidate 5/5: {'depth': 8, 'learning_rate': 0.02, 'l2_leaf_reg': 7.0, 'random_strength': 0.5}


,candidate,pinball_loss,best_iteration,MAE,RMSE,WAPE,R2,depth,learning_rate,l2_leaf_reg,random_strength
0,4,69.869861,1999,139.739721,405.422968,0.781640,-0.041809,7,0.05,5.0,0.5
1,3,73.098377,1999,146.196754,409.835591,0.817757,-0.064611,8,0.03,5.0,1.0
2,2,73.154526,1999,146.309053,409.939359,0.818386,-0.065150,7,0.03,3.0,1.0
3,1,73.225238,1999,146.450477,410.019114,0.819177,-0.065564,6,0.03,3.0,1.0
4,5,75.350419,1999,150.700838,412.271283,0.842951,-0.077303,8,0.02,7.0,0.5



BEST CATBOOST P50 CANDIDATE: 4
Pinball Loss: 69.8699
MAE: 139.7397
RMSE: 405.423
WAPE: 78.16%
R²: -0.0418


In [16]:
# ============================================================
# EXPERIMENT 4C — CatBoost P90 compact tuning
# ============================================================
cb_p90_candidates = [
    {
        'depth': 6,
        'learning_rate': 0.03,
        'l2_leaf_reg': 3.0,
        'random_strength': 1.0
    },
    {
        'depth': 7,
        'learning_rate': 0.03,
        'l2_leaf_reg': 3.0,
        'random_strength': 1.0
    },
    {
        'depth': 8,
        'learning_rate': 0.03,
        'l2_leaf_reg': 5.0,
        'random_strength': 1.0
    },
    {
        'depth': 7,
        'learning_rate': 0.05,
        'l2_leaf_reg': 5.0,
        'random_strength': 0.5
    }
]

cb_p90_rows = []
cb_p90_models = {}

for i, params in enumerate(cb_p90_candidates, start=1):
    print(
        f'CatBoost P90 candidate {i}/{len(cb_p90_candidates)}:',
        params
    )

    model = CatBoostRegressor(
        loss_function='Quantile:alpha=0.9',
        eval_metric='Quantile:alpha=0.9',
        iterations=2000,
        random_seed=RANDOM_STATE,
        task_type='GPU',
        devices='0',
        verbose=False,
        allow_writing_files=False,
        **params
    )

    model.fit(
        X_train_cb,
        y_train,
        cat_features=cat_feature_indices,
        eval_set=(X_val_cb, y_val),
        early_stopping_rounds=80,
        verbose=False
    )

    pred = np.clip(
        model.predict(X_val_cb),
        0,
        None
    )

    pinball = mean_pinball_loss(
        y_val,
        pred,
        alpha=0.90
    )

    cb_p90_rows.append({
        'candidate': i,
        'pinball_loss': pinball,
        'best_iteration': model.get_best_iteration(),
        **params
    })

    cb_p90_models[i] = model

cb_p90_results = (
    pd.DataFrame(cb_p90_rows)
    .sort_values('pinball_loss')
    .reset_index(drop=True)
)

display(cb_p90_results)

best_cb_p90_id = int(
    cb_p90_results.iloc[0]['candidate']
)

best_cb_p90_model = cb_p90_models[
    best_cb_p90_id
]

best_cb_p90_pred = np.clip(
    best_cb_p90_model.predict(X_val_cb),
    0,
    None
)

best_cb_p90_pinball = mean_pinball_loss(
    y_val,
    best_cb_p90_pred,
    alpha=0.90
)

print('\nBEST CATBOOST P90 CANDIDATE:', best_cb_p90_id)
print('Pinball Loss:', round(best_cb_p90_pinball, 4))


CatBoost P90 candidate 1/4: {'depth': 6, 'learning_rate': 0.03, 'l2_leaf_reg': 3.0, 'random_strength': 1.0}
CatBoost P90 candidate 2/4: {'depth': 7, 'learning_rate': 0.03, 'l2_leaf_reg': 3.0, 'random_strength': 1.0}
CatBoost P90 candidate 3/4: {'depth': 8, 'learning_rate': 0.03, 'l2_leaf_reg': 5.0, 'random_strength': 1.0}
CatBoost P90 candidate 4/4: {'depth': 7, 'learning_rate': 0.05, 'l2_leaf_reg': 5.0, 'random_strength': 0.5}


,candidate,pinball_loss,best_iteration,depth,learning_rate,l2_leaf_reg,random_strength
0,4,79.309763,1999,7,0.05,5.0,0.5
1,1,80.504537,1999,6,0.03,3.0,1.0
2,2,80.526252,1999,7,0.03,3.0,1.0
3,3,80.532265,1999,8,0.03,5.0,1.0



BEST CATBOOST P90 CANDIDATE: 4
Pinball Loss: 79.3098


In [17]:
# ============================================================
# EXPERIMENT 4D — CatBoost P10
# ============================================================
best_cb_p50_row = cb_p50_results.iloc[0]

cb_p10_model = CatBoostRegressor(
    loss_function='Quantile:alpha=0.1',
    eval_metric='Quantile:alpha=0.1',
    iterations=2000,
    depth=int(best_cb_p50_row['depth']),
    learning_rate=float(best_cb_p50_row['learning_rate']),
    l2_leaf_reg=float(best_cb_p50_row['l2_leaf_reg']),
    random_strength=float(best_cb_p50_row['random_strength']),
    random_seed=RANDOM_STATE,
    task_type='GPU',
    devices='0',
    verbose=False,
    allow_writing_files=False
)

cb_p10_model.fit(
    X_train_cb,
    y_train,
    cat_features=cat_feature_indices,
    eval_set=(X_val_cb, y_val),
    early_stopping_rounds=80,
    verbose=False
)

cb_p10_pred = np.clip(
    cb_p10_model.predict(X_val_cb),
    0,
    None
)

cb_p10_pinball = mean_pinball_loss(
    y_val,
    cb_p10_pred,
    alpha=0.10
)

print('CatBoost P10 Pinball:', round(cb_p10_pinball, 4))
print('Best iteration:', cb_p10_model.get_best_iteration())


CatBoost P10 Pinball: 17.5853
Best iteration: 1999


In [18]:
# ============================================================
# EXPERIMENT 4E — CatBoost interval evaluation
# ============================================================
cb_quantiles = np.column_stack([
    cb_p10_pred,
    best_cb_p50_pred,
    best_cb_p90_pred
])

cb_quantiles = np.sort(
    cb_quantiles,
    axis=1
)

cb_p10 = cb_quantiles[:, 0]
cb_p50 = cb_quantiles[:, 1]
cb_p90 = cb_quantiles[:, 2]

cb_p10_pinball = mean_pinball_loss(
    y_val,
    cb_p10,
    alpha=0.10
)

cb_p50_pinball = mean_pinball_loss(
    y_val,
    cb_p50,
    alpha=0.50
)

cb_p90_pinball = mean_pinball_loss(
    y_val,
    cb_p90,
    alpha=0.90
)

cb_p50_metrics = regression_metrics(
    y_val,
    cb_p50
)

cb_coverage = np.mean(
    (y_val >= cb_p10)
    & (y_val <= cb_p90)
)

cb_interval_width = np.mean(
    cb_p90 - cb_p10
)

print('CatBoost Quantile Validation Results')
print('P10 Pinball:', round(cb_p10_pinball, 4))
print('P50 Pinball:', round(cb_p50_pinball, 4))
print('P90 Pinball:', round(cb_p90_pinball, 4))
print('P50 MAE:', round(cb_p50_metrics['MAE'], 4))
print('P50 RMSE:', round(cb_p50_metrics['RMSE'], 4))
print('P50 WAPE:', f"{cb_p50_metrics['WAPE']*100:.2f}%")
print('P50 R²:', round(cb_p50_metrics['R2'], 4))
print('P10–P90 Coverage:', f'{cb_coverage*100:.2f}%')
print('Mean interval width:', round(cb_interval_width, 2))


CatBoost Quantile Validation Results
P10 Pinball: 17.5853
P50 Pinball: 69.8699
P90 Pinball: 79.3098
P50 MAE: 139.7397
P50 RMSE: 405.423
P50 WAPE: 78.16%
P50 R²: -0.0418
P10–P90 Coverage: 84.31%
Mean interval width: 494.35


In [19]:
# ============================================================
# SAVE — CatBoost Validation Results
# ============================================================
cb_summary = pd.DataFrame([
    {
        'Model': 'CatBoost Quantile',
        'P50_MAE': cb_p50_metrics['MAE'],
        'P50_RMSE': cb_p50_metrics['RMSE'],
        'P50_WAPE': cb_p50_metrics['WAPE'],
        'P50_R2': cb_p50_metrics['R2'],
        'P10_Pinball': cb_p10_pinball,
        'P50_Pinball': cb_p50_pinball,
        'P90_Pinball': cb_p90_pinball,
        'Coverage_P10_P90': cb_coverage,
        'Mean_Interval_Width': cb_interval_width
    }
])

display(cb_summary)

cb_summary.to_csv(
    OUTPUT_PATH / 'catboost_validation_metrics.csv',
    index=False
)

cb_p50_results.to_csv(
    OUTPUT_PATH / 'catboost_p50_tuning_results.csv',
    index=False
)

cb_p90_results.to_csv(
    OUTPUT_PATH / 'catboost_p90_tuning_results.csv',
    index=False
)

pred_df = val_df[
    ['hf_pk', 'productID', 'date_parsed', TARGET]
].copy()

pred_df['P10'] = cb_p10
pred_df['P50'] = cb_p50
pred_df['P90'] = cb_p90

pred_df.to_csv(
    OUTPUT_PATH / 'catboost_validation_predictions.csv',
    index=False
)

joblib.dump(
    best_cb_p50_model,
    OUTPUT_PATH / 'catboost_best_p50_model.joblib'
)

joblib.dump(
    best_cb_p90_model,
    OUTPUT_PATH / 'catboost_best_p90_model.joblib'
)

joblib.dump(
    cb_p10_model,
    OUTPUT_PATH / 'catboost_p10_model.joblib'
)

print('=' * 72)
print('MADAD — CATBOOST VALIDATION CHECKPOINT')
print('=' * 72)
print(f"P50 MAE: {cb_p50_metrics['MAE']:.4f}")
print(f"P90 Pinball: {cb_p90_pinball:.4f}")
print(f"Coverage: {cb_coverage*100:.2f}%")
print('TEST STATUS: UNTOUCHED')
print('Saved to:', OUTPUT_PATH)
print('=' * 72)


,Model,P50_MAE,P50_RMSE,P50_WAPE,P50_R2,P10_Pinball,P50_Pinball,P90_Pinball,Coverage_P10_P90,Mean_Interval_Width
0,CatBoost Quantile,139.739721,405.422968,0.78164,-0.041809,17.585263,69.869861,79.309763,0.843117,494.346077


MADAD — CATBOOST VALIDATION CHECKPOINT
P50 MAE: 139.7397
P90 Pinball: 79.3098
Coverage: 84.31%
TEST STATUS: UNTOUCHED
Saved to: /content/drive/MyDrive/Madad for Medical Logistics/Dataset/Dryad_Dataset/forecasting_outputs_clean/05_catboost_quantile
